In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import max, explode, col

from pyspark.ml.feature import StringIndexer

In [2]:
# Инициализируем Spark-сессию под единым именем
spark = SparkSession.builder \
    .appName("pyspark_als_demo") \
    .master("local[*]") \
    .getOrCreate()

## Чтение данных

In [3]:
books = spark.read.csv("work/data/book_rec/Books.csv", header=True, inferSchema=True)
ratings = spark.read.csv("work/data/book_rec/Ratings.csv", header=True, inferSchema=True)
users = spark.read.csv("work/data/book_rec/Users.csv", header=True, inferSchema=True)

In [4]:
books.show(5)

+----------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+
|      ISBN|          Book-Title|         Book-Author|Year-Of-Publication|           Publisher|         Image-URL-S|         Image-URL-M|         Image-URL-L|
+----------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+
|0195153448| Classical Mythology|  Mark P. O. Morford|               2002|Oxford University...|http://images.ama...|http://images.ama...|http://images.ama...|
|0002005018|        Clara Callan|Richard Bruce Wright|               2001|HarperFlamingo Ca...|http://images.ama...|http://images.ama...|http://images.ama...|
|0060973129|Decision in Normandy|        Carlo D'Este|               1991|     HarperPerennial|http://images.ama...|http://images.ama...|http://images.ama...|
|0374157065|Flu: The Story of...|    Gina Bari

In [5]:
ratings.show(5)

+-------+----------+-----------+
|User-ID|      ISBN|Book-Rating|
+-------+----------+-----------+
| 276725|034545104X|          0|
| 276726|0155061224|          5|
| 276727|0446520802|          0|
| 276729|052165615X|          3|
| 276729|0521795028|          6|
+-------+----------+-----------+
only showing top 5 rows



In [6]:
users.show(5)

+-------+--------------------+----+
|User-ID|            Location| Age|
+-------+--------------------+----+
|      1|  nyc, new york, usa|NULL|
|      2|stockton, califor...|18.0|
|      3|moscow, yukon ter...|NULL|
|      4|porto, v.n.gaia, ...|17.0|
|      5|farnborough, hant...|NULL|
+-------+--------------------+----+
only showing top 5 rows



## Холодный старт

популряность - наиболее продаваемые, но при этом для рекомендации нужно, чтобы были хорошие оценки  

поэтому будем использовать IMDB-style weighted rating, так как это позволит нам сразу учитывать и оценки и количество отзывов:

$$
\text{WR} = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
$$

где:
- R — средняя оценка книги,
- v — число оценок книги,
- m — порог,
- C — глобальное среднее по всем оценкам.


In [7]:
C = ratings.agg(F.avg("Book-Rating")).first()[0]

counts = ratings.groupBy("ISBN").agg(F.count("Book-Rating").alias("v"))
m = counts.approxQuantile("v", [0.75], 0.01)[0]

book_stats = (
    ratings.groupBy("ISBN")
           .agg(F.avg("Book-Rating").alias("R"),
                F.count("Book-Rating").alias("v"))
           .withColumn("weighted_rating",
                (F.col("v") / (F.col("v") + F.lit(m))) * F.col("R") +
                (F.lit(m) / (F.col("v") + F.lit(m))) * F.lit(C))
           .orderBy(F.desc("weighted_rating"))
)

In [8]:
book_stats.show(5)

+----------+-----------------+---+-----------------+
|      ISBN|                R|  v|  weighted_rating|
+----------+-----------------+---+-----------------+
|1563891336|9.444444444444445|  9|8.248536399848826|
|0395193958|             10.0|  6|8.216737549792134|
|0091842050|9.181818181818182| 11|8.210300030641314|
|0823401898|              9.5|  8|8.173390039833707|
|0385326335|              9.5|  8|8.173390039833707|
+----------+-----------------+---+-----------------+
only showing top 5 rows



In [9]:
book_stats.count()

340556

In [10]:
books.count()

271360

In [11]:
ratings.select("ISBN").distinct().count()

340556

странное наблюдение - книг в рейтингах больше, чем в самой таблице с книгами 

In [12]:
book_stats.select(max("v")).show()

+------+
|max(v)|
+------+
|  2502|
+------+



In [13]:
book_stats.orderBy(book_stats["v"].desc()).limit(10).show()

+----------+------------------+----+------------------+
|      ISBN|                 R|   v|   weighted_rating|
+----------+------------------+----+------------------+
|0971880107|1.0195843325339728|2502|1.0210598643763327|
|0316666343| 4.468725868725869|1295| 4.466255898533799|
|0385504209| 4.652321630804077| 883| 4.648286893105466|
|0060928336| 3.448087431693989| 732| 3.446503951496372|
|0312195516| 4.334716459197787| 723| 4.330667448825292|
|044023722X| 3.187017001545595| 647| 3.186030663171552|
|0679781587| 4.381846635367762| 639| 4.377119969420183|
|0142001740| 4.219512195121951| 615| 4.215127877468942|
|067976402X| 3.255700325732899| 614|3.2544381499973003|
|0671027360| 3.718430034129693| 586| 3.715533844214859|
+----------+------------------+----+------------------+



In [14]:
books.where(books["ISBN"] == "0971880107").show()

+----------+-----------+------------+-------------------+---------+--------------------+--------------------+--------------------+
|      ISBN| Book-Title| Book-Author|Year-Of-Publication|Publisher|         Image-URL-S|         Image-URL-M|         Image-URL-L|
+----------+-----------+------------+-------------------+---------+--------------------+--------------------+--------------------+
|0971880107|Wild Animus|Rich Shapero|               2004|  Too Far|http://images.ama...|http://images.ama...|http://images.ama...|
+----------+-----------+------------+-------------------+---------+--------------------+--------------------+--------------------+



In [15]:
books.where(books["ISBN"] == "0316666343").show(truncate=False)

+----------+-------------------------+------------+-------------------+-------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|ISBN      |Book-Title               |Book-Author |Year-Of-Publication|Publisher    |Image-URL-S                                                 |Image-URL-M                                                 |Image-URL-L                                                 |
+----------+-------------------------+------------+-------------------+-------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|0316666343|The Lovely Bones: A Novel|Alice Sebold|2002               |Little, Brown|http://images.amazon.com/images/P/0316666343.01.THUMBZZZ.jpg|http://images.amazon.com/images/P/0316666343.01

In [16]:
books.where(books["ISBN"] == "1563891336").show(truncate=False)

+----------+------------------------------+-----------+-------------------+---------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|ISBN      |Book-Title                    |Book-Author|Year-Of-Publication|Publisher|Image-URL-S                                                 |Image-URL-M                                                 |Image-URL-L                                                 |
+----------+------------------------------+-----------+-------------------+---------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|1563891336|Death: The High Cost of Living|Neil Gaiman|1994               |DC Comics|http://images.amazon.com/images/P/1563891336.01.THUMBZZZ.jpg|http://images.amazon.com/images/P/1563891336.01

есть подозрения, что при создании датасета, у некоторых книг был пятибальный рейтинг, а не 10-ти бальный.

## Горячий старт

### Фильтрация в лоб

In [17]:
import pyspark.sql.functions as F

user_counts = (
    ratings.groupBy("User-ID")
           .agg(F.count("Book-Rating").alias("n"))
           .orderBy(F.desc("n"))
)
user_counts.show(10)

+-------+-----+
|User-ID|    n|
+-------+-----+
|  11676|13602|
| 198711| 7550|
| 153662| 6109|
|  98391| 5891|
|  35859| 5850|
| 212898| 4785|
| 278418| 4533|
|  76352| 3367|
| 110973| 3100|
| 235105| 3067|
+-------+-----+
only showing top 10 rows



In [18]:
target_user = user_counts.first()["User-ID"]
target_user

11676

In [19]:
target_ratings = (
    ratings.filter(F.col("User-ID") == target_user)
           .select("ISBN", F.col("Book-Rating").alias("target_rating"))
)
target_ratings.show(3)

+---------------+-------------+
|           ISBN|target_rating|
+---------------+-------------+
|     9022906116|            7|
|\0432534220""""|            6|
|\2842053052""""|            7|
+---------------+-------------+
only showing top 3 rows



In [20]:
similarity = (
    ratings.alias("r")
           .join(target_ratings.alias("t"), on="ISBN")
           .filter(F.col("r.User-ID") != target_user)
           .withColumn("match",
                (F.col("r.Book-Rating") == F.col("t.target_rating")).cast("int"))
           .groupBy(F.col("r.User-ID").alias("other_user"))
           .agg(
               F.sum("match").alias("matches"),
               F.count("*").alias("common_books")
           )
           .filter(F.col("common_books") >= 5)
           .orderBy(F.desc("matches"), F.desc("common_books"))
)

similarity.show(10)


+----------+-------+------------+
|other_user|matches|common_books|
+----------+-------+------------+
|     35859|    376|        1073|
|     76352|    327|         919|
|    198711|    259|         855|
|    102967|    255|         706|
|    153662|    245|         842|
|     52584|    222|         625|
|     55492|    219|         605|
|    230522|    208|         619|
|    234623|    202|         540|
|     78783|    199|         591|
+----------+-------+------------+
only showing top 10 rows



In [21]:
top3 = [row["other_user"] for row in similarity.limit(3).collect()]
top3

[35859, 76352, 198711]

In [22]:
neighbors_high = (
    ratings.filter(
        F.col("User-ID").isin(top3) & (F.col("Book-Rating") >= 8)
    )
    .select("User-ID", "ISBN", "Book-Rating")
)

In [23]:
candidates = (
    neighbors_high
    .join(target_ratings.select("ISBN"), on="ISBN", how="left_anti")
    .groupBy("ISBN")
    .agg(
        F.count("*").alias("neighbors_who_liked"),
        F.avg("Book-Rating").alias("avg_rating")
    )
)

In [24]:
recommendations = (
    candidates
    .join(book_stats.select("ISBN", "weighted_rating", "v"), on="ISBN", how="left")
    .join(books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left")
    .orderBy(F.desc("weighted_rating"))
)

recommendations.select(
    "Book-Title", "Book-Author",
    "weighted_rating", "avg_rating",
    "neighbors_who_liked", "v"
).show(20, truncate=False)

+------------------------------------------------------------------------------------------------------------+-------------------------+-----------------+----------+-------------------+---+
|Book-Title                                                                                                  |Book-Author              |weighted_rating  |avg_rating|neighbors_who_liked|v  |
+------------------------------------------------------------------------------------------------------------+-------------------------+-----------------+----------+-------------------+---+
|Charlottes Web Special Read Along Edition                                                                   |E B White                |7.622316733056178|10.0      |1                  |4  |
|Fox in Socks (I Can Read It All by Myself Beginner Books)                                                   |Dr. Seuss                |6.796852399920813|10.0      |1                  |19 |
|NULL                                             

### ALS

In [25]:
ratings.groupBy("Book-Rating").count().orderBy("Book-Rating").show()

+-----------+------+
|Book-Rating| count|
+-----------+------+
|          0|716109|
|          1|  1770|
|          2|  2759|
|          3|  5996|
|          4|  8904|
|          5| 50974|
|          6| 36924|
|          7| 76457|
|          8|103736|
|          9| 67541|
|         10| 78610|
+-----------+------+



много строк с нулевыми оценками - такого не должно быть, так как минимальная оценка, которую можно поставить 1.

In [26]:
ratings_clean = ratings.filter(F.col("Book-Rating") > 0)
print("до:", ratings.count(), "после:", ratings_clean.count())

до: 1149780 после: 433671


In [27]:
ratings_clean.select("ISBN").distinct().count()

185973

In [28]:
books.select("ISBN").distinct().count()

271360

теперь в рейтингах меньше книг, что логичнее

In [29]:
(training, test) = ratings_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {training.count()}, Test: {test.count()}")

Train: 346954, Test: 86717


In [30]:
indexer = StringIndexer(
    inputCol="ISBN",
    outputCol="ISBN_idx",
    handleInvalid="skip"
)

In [31]:
indexer_model = indexer.fit(training)

In [32]:
training_idx = indexer_model.transform(training)
test_idx = indexer_model.transform(test)

In [33]:
training_idx.select("User-ID", "ISBN", "ISBN_idx", "Book-Rating").show(5)

+-------+----------+--------+-----------+
|User-ID|      ISBN|ISBN_idx|Book-Rating|
+-------+----------+--------+-----------+
|      8|0002005018|  5008.0|          5|
|      8|074322678X|100140.0|          5|
|      8|1552041778|123775.0|          5|
|      8|1567407781|126817.0|          6|
|      8|1575663937|128318.0|          6|
+-------+----------+--------+-----------+
only showing top 5 rows



In [34]:
als = ALS(
    maxIter=10,
    rank=20,
    regParam=0.1,
    userCol="User-ID",
    itemCol="ISBN_idx",
    ratingCol="Book-Rating",
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=False,
    seed=42
)


In [35]:
model = als.fit(training_idx)

In [36]:
predictions = model.transform(test_idx)

In [37]:
evaluator = RegressionEvaluator(
    metricName="rmse", labelCol="Book-Rating", predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)

mae = RegressionEvaluator(
    metricName="mae", labelCol="Book-Rating", predictionCol="prediction"
).evaluate(predictions)

r2 = RegressionEvaluator(
    metricName="r2", labelCol="Book-Rating", predictionCol="prediction"
).evaluate(predictions)

print(f"RMSE = {rmse:.4f} | MAE = {mae:.4f} | R² = {r2:.4f}")

RMSE = 2.5031 | MAE = 1.9611 | R² = -0.9257


попробуем еще отфильтровать от шума

In [38]:
MIN_U, MIN_B = 5, 5
active_users = ratings_clean.groupBy("User-ID").count().filter(F.col("count") >= MIN_U).select("User-ID")
popular_books = ratings_clean.groupBy("ISBN").count().filter(F.col("count") >= MIN_B).select("ISBN")

ratings_filtered = (
    ratings_clean
    .join(active_users, on="User-ID")
    .join(popular_books, on="ISBN")
)

In [39]:
train_filtered, test_filtered = ratings_filtered.randomSplit([0.8, 0.2], seed=42)

In [40]:
isbn_indexer = StringIndexer(inputCol="ISBN", outputCol="ISBN_idx", handleInvalid="skip")
isbn_indexer_model = isbn_indexer.fit(train_filtered)

training_idx_filtered = isbn_indexer_model.transform(train_filtered)
test_idx_filtered = isbn_indexer_model.transform(test_filtered)

In [41]:
als = ALS(
    maxIter=10, rank=20, regParam=0.1,
    userCol="User-ID", itemCol="ISBN_idx", ratingCol="Book-Rating",
    coldStartStrategy="drop", nonnegative=True, seed=42
)
model = als.fit(training_idx_filtered)

In [42]:
predictions_filtered = model.transform(test_idx_filtered)

In [43]:
evaluator = RegressionEvaluator(
    metricName="rmse", labelCol="Book-Rating", predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions_filtered)

mae = RegressionEvaluator(
    metricName="mae", labelCol="Book-Rating", predictionCol="prediction"
).evaluate(predictions_filtered)

r2 = RegressionEvaluator(
    metricName="r2", labelCol="Book-Rating", predictionCol="prediction"
).evaluate(predictions_filtered)

print(f"RMSE = {rmse:.4f} | MAE = {mae:.4f} | R² = {r2:.4f}")

RMSE = 0.9209 | MAE = 0.4760 | R² = 0.7321


In [44]:
K = 10

top_k = model.recommendForAllUsers(K)

top_k_flat = (
    top_k
    .withColumn("rec", F.explode("recommendations"))
    .select("User-ID", F.col("rec.ISBN_idx").alias("ISBN_idx"))
)

relevant = (
    test_idx.filter(F.col("Book-Rating") >= 8)
            .select("User-ID", "ISBN_idx")
)

hits = top_k_flat.join(relevant, on=["User-ID", "ISBN_idx"], how="inner")

precision_at_k = (
    hits.groupBy("User-ID").count()
         .join(top_k_flat.groupBy("User-ID").count().withColumnRenamed("count", "k"),
               on="User-ID")
         .withColumn("p_at_k", F.col("count") / F.col("k"))
         .agg(F.avg("p_at_k"))
         .first()[0]
)
print(f"Precision@{K} = {precision_at_k:.4f}")

Precision@10 = 0.1000


In [45]:
isbn_map = (
    training_idx_filtered
    .select("ISBN", "ISBN_idx")
    .distinct()
)

In [46]:
user_recs = model.recommendForAllUsers(10)

recs_flat = (
    user_recs
    .withColumn("rec", explode("recommendations"))
    .select(
        "User-ID",
        col("rec.ISBN_idx").alias("ISBN_idx"),
        col("rec.rating").alias("predicted_rating")
    )
)

In [47]:
recs_named = (
    recs_flat
    .join(isbn_map, on="ISBN_idx", how="left")                       # → ISBN
    .join(
        books.select("ISBN", "Book-Title", "Book-Author",
                     "Year-Of-Publication", "Publisher"),
        on="ISBN", how="left"
    )                                                                  # → текст
)

In [48]:
recs_named.select(
    "User-ID",
    "Book-Title",
    "Book-Author",
    "Year-Of-Publication",
    F.round("predicted_rating", 2).alias("predicted")
).orderBy("User-ID", F.desc("predicted")).show(20, truncate=False)

+-------+--------------------------------------------------------------------+-------------------+-------------------+---------+
|User-ID|Book-Title                                                          |Book-Author        |Year-Of-Publication|predicted|
+-------+--------------------------------------------------------------------+-------------------+-------------------+---------+
|8      |LA Insoportable Levedad Del Ser/the Unbearable Lightness of Being   |Milan Kundera      |1993               |6.6      |
|8      |Le Cycle d'Ender, tome 2 : La Voix des morts                        |Scott Card Orson   |2001               |6.55     |
|8      |1984 (Spanish Language Edition)                                     |George Orwell      |1984               |6.53     |
|8      |Schande                                                             |J. M. Coetzee      |2002               |6.46     |
|8      |Per Anhalter durch die Galaxis.                                     |Douglas Adams      

## Гибрид

In [58]:
top_3_popular = (
    book_stats
    .orderBy(F.desc("weighted_rating"))
    .limit(3)
    .select("ISBN")
)

In [59]:
old_users = training_idx_filtered.select("User-ID").distinct()

In [60]:
all_users = ratings.select("User-ID").distinct()

In [61]:
cold_users = all_users.join(old_users, on="User-ID", how="left_anti")

In [62]:
cold_recs = (
    cold_users
    .crossJoin(top_3_popular)
    .withColumn("user_type", F.lit("cold"))
)

In [63]:
als_recs = model.recommendForAllUsers(3)
als_recs_flat = (
    als_recs
    .withColumn("rec", F.explode("recommendations"))
    .select("User-ID", F.col("rec.ISBN_idx").alias("ISBN_idx"))
)

In [64]:
isbn_map = training_idx_filtered.select("ISBN", "ISBN_idx").distinct()
als_recs_isbn = (
    als_recs_flat
    .join(isbn_map, on="ISBN_idx", how="left")
    .select("User-ID", "ISBN")
    .withColumn("user_type", F.lit("old"))
)

In [65]:
hybrid_recs = als_recs_isbn.unionByName(cold_recs)

In [66]:
final_hybrid_recs = (
    hybrid_recs
    .join(
        books.select("ISBN", "Book-Title", "Book-Author"), 
        on="ISBN", how="left"
    )
    .select("User-ID", "user_type", "ISBN", "Book-Title", "Book-Author")
    .orderBy("user_type", "User-ID")
)

In [68]:
final_hybrid_recs.show(30, truncate=False)

+-------+---------+----------+-------------------------------------------------------+---------------------+
|User-ID|user_type|ISBN      |Book-Title                                             |Book-Author          |
+-------+---------+----------+-------------------------------------------------------+---------------------+
|2      |cold     |0395193958|The Lord of the Rings (Leatherette Collector's Edition)|J. R. R. Tolkien     |
|2      |cold     |0091842050|The Blue Day Book: A Lesson in Cheering Yourself Up    |Bradley Trevor Greive|
|2      |cold     |1563891336|Death: The High Cost of Living                         |Neil Gaiman          |
|7      |cold     |0091842050|The Blue Day Book: A Lesson in Cheering Yourself Up    |Bradley Trevor Greive|
|7      |cold     |0395193958|The Lord of the Rings (Leatherette Collector's Edition)|J. R. R. Tolkien     |
|7      |cold     |1563891336|Death: The High Cost of Living                         |Neil Gaiman          |
|9      |cold     |

In [69]:
hybrid_recs.groupBy("user_type").agg(
    F.countDistinct("User-ID").alias("unique_users"),
    F.count("ISBN").alias("total_recs_issued")
).show()

+---------+------------+-----------------+
|user_type|unique_users|total_recs_issued|
+---------+------------+-----------------+
|      old|       13020|            39060|
|     cold|       92269|           276807|
+---------+------------+-----------------+



Для холодных книг

Exploration / Multi-Armed Bandits (Исследование): Внедрить ϵ-greedy стратегию или UCB (Upper Confidence Bound). Система будет намеренно "подмешивать" в выдачу небольшой процент новых книг (например, 5% трафика), чтобы собрать первые клики и оценки. Как только книга наберет минимальный пул реакций, ALS сможет ее подхватить.

Content-Based Similarity (Похожие по смыслу): Использовать NLP (TF-IDF, Word2Vec, BERT) для векторизации названий и описаний книг. Если вышла новая книга Стивена Кинга, она будет иметь высокую косинусную близость к его старым бестселлерам. Мы можем показывать новинку тем, кто высоко оценил его прошлые работы.

Эвристика "Новинки любимых авторов": Если пользователь фанат Джоан Роулинг, а у нее вышла новая книга (которой еще нет в ALS), мы можем рекомендовать ее в обход коллаборативной фильтрации, опираясь на связь User -> Author.

Отдельные полки "Тренды" и "Новые поступления": Ранжировать холодные книги не по историческим оценкам, а по свежим поведенческим метрикам: CTR (кликабельность обложки), количество добавлений в "Избранное" за последние 24 часа или скорость продаж.